# LeakAudit - 02 - Structural audit (E1-E3)

**Three checks that need no model training, only the metadata.**

- **E1** Target proxy: is the device field `lrate` a near-deterministic
  function of the label?
- **E2** Split contamination: do the official train/val folds share sites/sensors?
- **E3** Identity-only predictor: how far does sensor identity alone get us?


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # make 'src' importable from notebooks/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation
cfg.apply_mpl_style()
from sklearn.metrics import accuracy_score
X1, X1_max, M1, HZ = io_utils.load_d1()
RESULTS = {}

## E1 - Target leakage in device-derived fields

`lrate` is the logger's own leak-probability output, generated automatically by
the device. If it separates the classes almost perfectly, any model that
includes it will score highly for the wrong reason (Kapoor & Narayanan 2023,
category L2). The dataset documentation does not state that the `leaktype`
label was *derived* from `lrate`, so we describe this as a target-proxy
shortcut rather than asserting outright label leakage.

In [2]:
tab = M1.groupby("cls")[["lrate", "llevel"]].agg(
    ["min", "max", "mean", "std", "nunique"])
print(tab.round(2).to_string())

truth = np.where(M1.cls.values == "normal", "normal", "alarm")
best = None
for thr in sorted(M1.lrate.unique()):
    pred = np.where(M1.lrate.values >= thr, "alarm", "normal")
    acc = accuracy_score(truth, pred)
    if best is None or acc > best[1]:
        best = (thr, acc)
thr, acc = best
print(f"\nBest rule [lrate >= {thr}] -> alarm : accuracy = {acc:.6f}")
print(pd.crosstab(truth, np.where(M1.lrate.values >= thr, "alarm", "normal")).to_string())

verdict = "NEAR-DETERMINISTIC TARGET PROXY" if acc > 0.999 else "no strong proxy"
print("\nverdict:", verdict)
RESULTS["E1"] = dict(rule_threshold=float(thr), rule_accuracy=float(acc), verdict=verdict)
la.save_json(RESULTS["E1"], "E1_target_leakage.json")

       lrate                          llevel                              
         min max  mean    std nunique    min   max    mean     std nunique
cls                                                                       
in        90  90  90.0   0.00       1    114  1653  504.62  229.50    1037
noise     90  90  90.0   0.00       1    112  1274  586.30  264.93    1051
normal    30  70  45.3  14.11       5    125  1113  276.00  154.43     820
other     90  90  90.0   0.00       1    131  1212  520.87  221.16     932
out       90  90  90.0   0.00       1    134  1254  526.66  224.04    1008

Best rule [lrate >= 90] -> alarm : accuracy = 1.000000
col_0   alarm  normal
row_0                
alarm   48218       0
normal      0   22166

verdict: NEAR-DETERMINISTIC TARGET PROXY


WindowsPath('C:/Users/ecredible/Desktop/LeakAudit/LeakAudit/results/E1_target_leakage.json')

## E2 - Contamination of the official train/validation split

If most validation rows come from sites (or even sensors) also present in
training, the official split measures memorisation, not generalisation
(Kapoor & Narayanan 2023, category L3.2).

In [3]:
tr, va = M1[M1.split == "train"], M1[M1.split == "val"]
site_rate = float(va.site.isin(set(tr.site)).mean())
sid_rate  = float(va.sid.isin(set(tr.sid)).mean())
print(f"validation rows whose SITE   appears in training : {site_rate*100:6.2f}%")
print(f"validation rows whose SENSOR appears in training : {sid_rate*100:6.2f}%")
RESULTS["E2"] = dict(site_overlap_rate=site_rate, sid_overlap_rate=sid_rate)
la.save_json(RESULTS["E2"], "E2_split_contamination.json")

validation rows whose SITE   appears in training : 100.00%
validation rows whose SENSOR appears in training :  99.50%


WindowsPath('C:/Users/ecredible/Desktop/LeakAudit/LeakAudit/results/E2_split_contamination.json')

## E3 - Identity-only predictor

Predict the class from sensor identity alone, ignoring the signal. If this
already approaches model accuracy, then any model that can infer identity from
the waveform will look accurate without learning leak physics.

In [4]:
maj_sid  = M1.groupby("sid").cls.agg(lambda s: s.value_counts().idxmax())
acc_sid  = accuracy_score(M1.cls, M1.sid.map(maj_sid))
maj_site = M1.groupby("site").cls.agg(lambda s: s.value_counts().idxmax())
acc_site = accuracy_score(M1.cls, M1.site.map(maj_site))
base = M1.cls.value_counts(normalize=True).max()

ct = pd.crosstab(M1.sid, M1.cls); purity = (ct.max(1) / ct.sum(1))
print(f"global majority baseline : {base:.4f}")
print(f"site-identity accuracy   : {acc_site:.4f}")
print(f"sensor-identity accuracy : {acc_sid:.4f}")
print(f"single-class sensors     : {(purity > 0.99).sum()} / {len(purity)}")
RESULTS["E3"] = dict(majority_baseline=float(base),
                     acc_site_identity=float(acc_site),
                     acc_sensor_identity=float(acc_sid),
                     n_pure_sensors=int((purity > 0.99).sum()),
                     n_sensors=int(len(purity)))
la.save_json(RESULTS["E3"], "E3_identity_predictor.json")

global majority baseline : 0.3149
site-identity accuracy   : 0.6550
sensor-identity accuracy : 1.0000
single-class sensors     : 2710 / 2710


C:\Users\ecredible\AppData\Local\Temp\ipykernel_24112\3795251941.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of max will be keyword-only.
  ct = pd.crosstab(M1.sid, M1.cls); purity = (ct.max(1) / ct.sum(1))
C:\Users\ecredible\AppData\Local\Temp\ipykernel_24112\3795251941.py:7: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  ct = pd.crosstab(M1.sid, M1.cls); purity = (ct.max(1) / ct.sum(1))


WindowsPath('C:/Users/ecredible/Desktop/LeakAudit/LeakAudit/results/E3_identity_predictor.json')